# Inverse Physics-Informed Neural Network for Black–Scholes Parameter Estimation

## Objective

We use synthetic European call option prices to estimate the unknown constant
parameters in the Black–Scholes PDE:

The option price function (V(t,S)) is approximated by a neural network.
The volatility and risk-free rate are treated as unknown trainable constants.

For the synthetic experiment, option prices are generated using known true parameters. These true values are used only to generate and evaluate the data;
they will not be provided to the inverse PINN during training.

In [50]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from inverse_pinn_model import InversePINN

In [51]:
device = torch.device("cpu")
dtype = torch.float64

In [52]:
# Known contract parameters

K = 40.0
T = 1.0

# PINN domain
t_min = 0.0
t_max = T

S_min = 0.0
S_max = 160.0

In [53]:
# read csv

data_path = Path("data") / "synthetic_european_call_data_2000.csv"

synthetic_df = pd.read_csv(data_path)

print("Dataset shape:", synthetic_df.shape)
print("Columns:", synthetic_df.columns.tolist())

synthetic_df.head(10)

Dataset shape: (2000, 3)
Columns: ['t', 'S', 'V']


,t,S,V
0,0.878904,65.789270,26.030731
1,0.443117,46.604435,8.051128
2,0.653018,66.463420,27.151398
3,0.534206,92.272250,53.193070
4,0.006199,114.922005,76.861040
5,0.032571,33.990746,1.227104
6,0.878546,0.112338,0.000000
7,0.333371,125.848625,87.159910
8,0.240722,1.695512,0.000000
9,0.051166,78.728670,40.582280


In [54]:
# exchange to pytorch tensor

t_obs = torch.tensor(
    synthetic_df["t"].to_numpy(),
    dtype=dtype,
    device=device
).reshape(-1, 1)

S_obs = torch.tensor(
    synthetic_df["S"].to_numpy(),
    dtype=dtype,
    device=device
).reshape(-1, 1)

V_obs = torch.tensor(
    synthetic_df["V"].to_numpy(),
    dtype=dtype,
    device=device
).reshape(-1, 1)

In [55]:
print("Finite t:", torch.isfinite(t_obs).all().item())
print("Finite S:", torch.isfinite(S_obs).all().item())
print("Finite V:", torch.isfinite(V_obs).all().item())

print("\nt range:")
print(t_obs.min().item(), t_obs.max().item())

print("\nS range:")
print(S_obs.min().item(), S_obs.max().item())

print("\nV range:")
print(V_obs.min().item(), V_obs.max().item())

print("\nAny negative option prices:")
print((V_obs < 0).any().item())

Finite t: True
Finite S: True
Finite V: True

t range:
0.00064098276 0.9990846

S range:
0.0039528394 159.90704

V range:
-3.0914634e-07 121.74415

Any negative option prices:
True


In [56]:
N_obs = t_obs.shape[0]

# Randomly shuffle all row indices
indices = torch.randperm(N_obs)

# 80% training, 20% validation
N_train = int(0.8 * N_obs)

train_indices = indices[:N_train]
validation_indices = indices[N_train:]

# Training observations
t_train = t_obs[train_indices]
S_train = S_obs[train_indices]
V_train = V_obs[train_indices]

# Validation observations
t_validation = t_obs[validation_indices]
S_validation = S_obs[validation_indices]
V_validation = V_obs[validation_indices]

print("Training data:")
print(t_train.shape, S_train.shape, V_train.shape)

print("\nValidation data:")
print(
    t_validation.shape,
    S_validation.shape,
    V_validation.shape
)

Training data:
torch.Size([1600, 1]) torch.Size([1600, 1]) torch.Size([1600, 1])

Validation data:
torch.Size([400, 1]) torch.Size([400, 1]) torch.Size([400, 1])


In [57]:
u_train = V_train / K
u_validation = V_validation / K

In [58]:
def inverse_softplus(value):
    """
    Find a raw value x such：
    softplus(x) = value
    This let us initialize a positive trainable parameter at an interpretable value
    sigma可以是任何书，但是让模型训练，不管sigma_raw 多少，sigma_est都是post
    """
    if value <= 0:
        raise ValueError("Initial value must be positive.")

    value_tensor = torch.tensor(
        value,
        dtype=dtype,
        device=device
    )

    return torch.log(torch.expm1(value_tensor))

In [59]:
model = InversePINN(
    t_min=t_min,
    t_max=t_max,
    S_min=S_min,
    S_max=S_max,
    hidden_width=64,
    hidden_layers=4,
    r_initial=0.10,
    sigma_initial=0.40
).to(
    device=device,
    dtype=dtype
)

print(model)

InversePINN(
  (model): Sequential(
    (0): Linear(in_features=2, out_features=64, bias=True)
    (1): Tanh()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): Tanh()
    (4): Linear(in_features=64, out_features=64, bias=True)
    (5): Tanh()
    (6): Linear(in_features=64, out_features=64, bias=True)
    (7): Tanh()
    (8): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [60]:
# generate collocation points for the PDE residual (用来算residual的)
N_f = 2000
epsilon = 1e-4

# Random interior time points
t_f = (
    t_min + epsilon
    + (t_max - t_min - 2.0 * epsilon)
    * torch.rand(
        N_f,
        1,
        dtype=dtype,
        device=device
    )
)

# Random interior spot-price points
S_f = (
    S_min + epsilon
    + (S_max - S_min - 2.0 * epsilon)
    * torch.rand(
        N_f,
        1,
        dtype=dtype,
        device=device
    )
)

In [61]:
# define residual
def black_scholes_residual(
        model: nn.Module,
        t: torch.Tensor,
        S: torch.Tensor
):
    """
    Compute the black_scholes PDE residual for the normalized option price u(t,S) = V(t,S)/K

    Returns
        residual
        u: model prediction
        u_t: first derivaive with respect to time
        u_S: first derivative with respect to spot price
        u_SS: second derivative with respect to spot price
    """
    t_input = (
        t.clone()
        .detach().requires_grad_(True)
    )
    S_input = (
        S.clone()
        .detach()
        .requires_grad_(True)
    )

    # predict normalized option price
    u = model(t_input, S_input)

    # first derivative / t
    u_t = torch.autograd.grad(
        outputs = u,
        inputs = t_input,
        grad_outputs=torch.ones_like(u),        # weight = 1
        create_graph = True,
        retain_graph= True
    )[0]

    # first derivative / S
    u_S = torch.autograd.grad(
        outputs = u,
        inputs = S_input,
        grad_outputs=torch.ones_like(u),
        create_graph=True,
        retain_graph=True
    )[0]

    # second derivative / S
    u_SS = torch.autograd.grad(
        outputs=u_S,
        inputs=S_input,
        grad_outputs=torch.ones_like(u_S),
        create_graph=True,
        retain_graph=True
    )[0]

    # current trainable parameter estimates
    r_est = model.r
    sigma_est = model.sigma

    # black-scholes PDE residual
    residual = (
        u_t
        + 0.5
        * sigma_est**2
        * S_input**2
        * u_SS
        + r_est
        * S_input
        * u_S
        - r_est
        * u
    )

    return residual, u, u_t, u_S, u_SS

In [62]:
# generate terminal/boundary points
# 在区间里面随机生成数

N_terminal = 500
N_boundary = 500

# terminal-condition points: t=T
t_terminal = torch.full(
    (N_terminal, 1),
    fill_value=T,
    dtype=dtype,
    device=device
)

S_terminal = (
    S_min + (S_max - S_min) * torch.rand(
        N_terminal,
        1,
        dtype = dtype,
        device = device
    )
)

# boundary points: random times between 0 and T
t_boundary = (
    t_min + (t_max - t_min) * torch.rand(
        N_boundary,
        1,
        dtype = dtype,
        device = device
    )
)

# left boundary: S = S_min = 0
S_boundary_left = torch.full(
    (N_boundary, 1),
    fill_value = S_min,
    dtype=dtype,
    device=device
)

# rigt boundary: S = S_max = 160
S_boundary_right = torch.full(
    (N_boundary, 1),
    fill_value=S_max,
    dtype=dtype,
    device=device
)


In [63]:
# define loss function
def compute_losses(
    model,
    t_train,
    S_train,
    u_train,
    t_f,
    S_f,
    t_terminal,
    S_terminal,
    t_boundary,
    S_boundary_left,
    S_boundary_right,
    lambda_data=6.0,
    lambda_pde=1.50,
    lambda_terminal=1.0,
    lambda_boundary=1.0
):
    """
    Compute all loss components for the inverse PINN
    """
    # 1. data residual
    u_data_prediction = model(
        t_train,
        S_train
    )
    loss_data = torch.mean(
        (u_data_prediction - u_train) **2
    )

    # 2. PDE loss （检查是否满足pde = 0）
    # pde prediction is left hand side of pde
    pde_prediction, _, _, _, _ = (
        black_scholes_residual(
            model,
            t_f,
            S_f
        )
    )

    pde_true = torch.zeros_like(
        pde_prediction
    )
    residual_pde=(
        pde_true - pde_prediction
    )
    loss_pde=torch.mean(
        residual_pde **2
    )


    # 3. Terminal residual
    # t = T, 预测价格是非等于European call payoff
    u_terminal_pred = model(
        t_terminal,
        S_terminal
    )
    u_terminal_true=(
        torch.relu(S_terminal - K) / K
    )
    residual_terminal = (
        u_terminal_true - u_terminal_pred
    )
    loss_terminal = torch.mean(
        residual_terminal **2
    )

    # 4. Left-boundary residual
    u_left_pred = model(
        t_boundary,
        S_boundary_left
    )
    u_left_true = torch.zeros_like(
        u_left_pred
    )
    residual_left=(
        u_left_true - u_left_pred
    )
    loss_boundary_left = torch.mean(
        residual_left **2
    )

    # 5。 Right boundary residual
    u_right_pred = model(
        t_boundary,
        S_boundary_right
    )

    u_right_true = (
        S_boundary_right
        - K * torch.exp(
            -model.r * (T - t_boundary)
        )
    ) / K

    residual_right = (
        u_right_true
        - u_right_pred
    )

    loss_boundary_right = torch.mean(
        residual_right ** 2
    )

    loss_boundary = (
        loss_boundary_left
        + loss_boundary_right
    )

    # Total loss
    total_loss = (
        lambda_data * loss_data
        + lambda_pde * loss_pde
        + lambda_terminal * loss_terminal
        + lambda_boundary * loss_boundary
    )

    return {
        "total": total_loss,
        "data": loss_data,
        "pde": loss_pde,
        "terminal": loss_terminal,
        "boundary": loss_boundary,
        "boundary_left": loss_boundary_left,
        "boundary_right": loss_boundary_right
    }

In [64]:
# Calculate the losses before training

initial_losses = compute_losses(
    model=model,

    # Synthetic observed data
    t_train=t_train,
    S_train=S_train,
    u_train=u_train,

    # PDE collocation points
    t_f=t_f,
    S_f=S_f,

    # Terminal-condition points
    t_terminal=t_terminal,
    S_terminal=S_terminal,

    # Boundary-condition points
    t_boundary=t_boundary,
    S_boundary_left=S_boundary_left,
    S_boundary_right=S_boundary_right,

    # Initial loss weights
    lambda_data=6.0,
    lambda_pde=1.50,
    lambda_terminal=1.0,
    lambda_boundary=1.0
)

print("Initial losses before training")

for loss_name, loss_value in initial_losses.items():
    print(
        f"{loss_name:16s}: "
        f"{loss_value.item():.8f}"
    )

Initial losses before training
total           : 22.68927465
data            : 2.07574091
pde             : 0.00584211
terminal        : 2.18527992
boundary        : 8.04078611
boundary_left   : 0.04952825
boundary_right  : 7.99125787


In [65]:
# training configuration
learning_rate = 1e-3
num_epochs = 30000
print_every = 500

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate
)

In [66]:
# store training result
    # gonna be used for loss curve and parameter path
history = {
    "epoch": [],
    "total_loss": [],
    "data_loss": [],
    "pde_loss": [],
    "terminal_loss": [],
    "boundary_loss": [],
    "validation_loss": [],
    "r_estimate": [],
    "sigma_estimate": []
}

In [67]:
# training loop
for epoch in range(1, num_epochs + 1):

    # Put the model in training mode
    model.train()

    # Remove gradients left from the previous epoch
    optimizer.zero_grad(set_to_none=True)

    # Calculate all loss components
    losses = compute_losses(
        model=model,

        # Synthetic observed prices
        t_train=t_train,
        S_train=S_train,
        u_train=u_train,

        # PDE collocation points
        t_f=t_f,
        S_f=S_f,

        # Terminal-condition points
        t_terminal=t_terminal,
        S_terminal=S_terminal,

        # Boundary-condition points
        t_boundary=t_boundary,
        S_boundary_left=S_boundary_left,
        S_boundary_right=S_boundary_right,

        # Loss weights
        lambda_data=6.0,
        lambda_pde=1.50,
        lambda_terminal=1.0,
        lambda_boundary=1.0
    )

    # compute the gradients of total loss
    losses["total"].backward()
    # update NN weights, r_raw, and sigma_raw
    optimizer.step()

    # validation-price loss
    model.eval()
    with torch.no_grad():
        u_validation_prediction = model(
            t_validation,
            S_validation
        )
        validation_loss = torch.mean(
            (
                u_validation - u_validation_prediction
            ) **2
        )


    # save training history
    history["epoch"].append(epoch)

    history["total_loss"].append(
        losses["total"].item()
    )

    history["data_loss"].append(
        losses["data"].item()
    )

    history["pde_loss"].append(
        losses["pde"].item()
    )

    history["terminal_loss"].append(
        losses["terminal"].item()
    )

    history["boundary_loss"].append(
        losses["boundary"].item()
    )

    history["validation_loss"].append(
        validation_loss.item()
    )

    history["r_estimate"].append(
        model.r.item()
    )

    history["sigma_estimate"].append(
        model.sigma.item()
    )

    # print the progress
    if epoch == 1 or epoch % print_every == 0:

        print(
            f"Epoch {epoch:5d} | "
            f"Total: {losses['total'].item():.6e} | "
            f"Data: {losses['data'].item():.6e} | "
            f"PDE: {losses['pde'].item():.6e} | "
            f"Terminal: {losses['terminal'].item():.6e} | "
            f"Boundary: {losses['boundary'].item():.6e} | "
            f"Val: {validation_loss.item():.6e} | "
            f"r: {model.r.item():.6f} | "
            f"sigma: {model.sigma.item():.6f}"
        )

Epoch     1 | Total: 2.268927e+01 | Data: 2.075741e+00 | PDE: 5.842113e-03 | Terminal: 2.185280e+00 | Boundary: 8.040786e+00 | Val: 1.803055e+00 | r: 0.099905 | sigma: 0.399670
Epoch   500 | Total: 5.230417e-02 | Data: 4.865440e-03 | PDE: 5.434537e-03 | Terminal: 5.584913e-03 | Boundary: 9.374809e-03 | Val: 4.734049e-03 | r: 0.085048 | sigma: 0.280621
Epoch  1000 | Total: 5.338240e-03 | Data: 2.402086e-04 | PDE: 1.996662e-03 | Terminal: 5.063245e-04 | Boundary: 3.956703e-04 | Val: 2.747122e-04 | r: 0.080043 | sigma: 0.243282
Epoch  1500 | Total: 1.996934e-03 | Data: 1.673080e-04 | PDE: 3.478115e-04 | Terminal: 3.678930e-04 | Boundary: 1.034761e-04 | Val: 1.726590e-04 | r: 0.076527 | sigma: 0.232563
Epoch  2000 | Total: 2.949100e-03 | Data: 1.729803e-04 | PDE: 8.962238e-04 | Terminal: 3.609474e-04 | Boundary: 2.059349e-04 | Val: 1.484294e-04 | r: 0.072755 | sigma: 0.223795
Epoch  2500 | Total: 1.370800e-03 | Data: 1.258640e-04 | PDE: 1.267582e-04 | Terminal: 3.188418e-04 | Boundary: 1.0

In [68]:
# final para est

r_estimated = model.r.item()
sigma_estimated = model.sigma.item()

print("Final estimated parameters")
print(f"Estimated r: {r_estimated:.8f}")
print(f"Estimated sigma: {sigma_estimated:.8f}")

Final estimated parameters
Estimated r: 0.05047583
Estimated sigma: 0.18538445


In [69]:
# compare estimates with the synthetic truth
r_true = 0.05
sigma_true = 0.20

r_absolute_error = abs(
    r_estimated - r_true
)

sigma_absolute_error = abs(
    sigma_estimated - sigma_true
)

r_relative_error = (
    r_absolute_error / abs(r_true)
)

sigma_relative_error = (
    sigma_absolute_error / abs(sigma_true)
)

print(f"True r: {r_true:.8f}")
print(f"Estimated r: {r_estimated:.8f}")
print(f"Absolute error: {r_absolute_error:.8f}")
print(f"Relative error: {100 * r_relative_error:.4f}%")

print()

print(f"True sigma: {sigma_true:.8f}")
print(f"Estimated sigma: {sigma_estimated:.8f}")
print(f"Absolute error: {sigma_absolute_error:.8f}")
print(f"Relative error: {100 * sigma_relative_error:.4f}%")

True r: 0.05000000
Estimated r: 0.05047583
Absolute error: 0.00047583
Relative error: 0.9517%

True sigma: 0.20000000
Estimated sigma: 0.18538445
Absolute error: 0.01461555
Relative error: 7.3078%


In [73]:
# True PDE Solution
# Compare the trained PINN with the analytical 
# Black-Scholes solution on a uniform 100* 100 test grid

# 1. Analytical Black-Scholes Solution
def black_scholes_call(
        t,
        S,
        K,
        T,
        r, 
        sigma
):
    # time remaining until maturity
    tau = T - t
    eps = 1e-12
    tau_safe = torch.clamp(
        tau,
        min=eps
    )

    S_safe = torch.clamp(
        S,
        min=eps
    )


    # d1
    d1 = (
        torch.log(S_safe / K)
        + (
            r
            + 0.5 * sigma**2
        ) * tau_safe
    ) / (
        sigma * torch.sqrt(tau_safe)
    )


    # d2
    d2 = (
        d1
        - sigma * torch.sqrt(tau_safe)
    )


    # Standard normal CDF
    N_d1 = 0.5 * (
        1.0
        + torch.erf(
            d1 / np.sqrt(2.0)
        )
    )

    N_d2 = 0.5 * (
        1.0
        + torch.erf(
            d2 / np.sqrt(2.0)
        )
    )


    # Black-Scholes European call price
    V = (
        S_safe * N_d1
        - K
        * torch.exp(
            -r * tau_safe
        )
        * N_d2
    )


    # Exact terminal condition
    payoff = torch.relu(
        S - K
    )

    V = torch.where(
        tau <= eps,
        payoff,
        V
    )


    # Exact left boundary
    V = torch.where(
        S <= eps,
        torch.zeros_like(V),
        V
    )


    return V

# Build a uniform 100* 100 test mesh
N_test = 100

t_test_1d = torch.linspace(
    t_min,
    t_max,
    N_test,
    device=device,
    dtype=dtype
)

S_test_1d = torch.linspace(
    S_min,
    S_max,
    N_test,
    device=device,
    dtype=dtype
)

t_test_mesh, S_test_mesh = torch.meshgrid(
    t_test_1d,
    S_test_1d,
    indexing="ij"
)

# flatten
t_test = t_test_mesh.reshape(-1,1)
S_test = S_test_mesh.reshape(-1,1)

# Compute the true PDE Solution
with torch.no_grad():
    V_true = black_scholes_call(
        t = t_test,
        S = S_test,
        K = K,
        T = T,
        r = r_true,
        sigma = sigma_true

    )

# Compute PINN solution
model.eval()

with torch.no_grad():
    u_pinn = model(
        t_test,
        S_test,
    )
    # model predicts normalized price u = V/K
    V_pinn = K * u_pinn

# Compare true PDE solution and PINN solution
pde_test_mse = torch.mean((V_true - V_pinn) ** 2).item()

V_true_np = (
    V_true.cpu().numpy().reshape(-1)
)

V_pinn_np = (
    V_pinn.cpu().numpy().reshape(-1)
)

pde_test_correlation = np.corrcoef(
    V_true_np, V_pinn_np
)[0,1]


# report results
print("True PDE Solution vs. PINN")
print(
    f"MSE(V_true, V_PINN)"
    f"{pde_test_mse:.8f}"
)
print(
    f"cor(V_true, V_PINN): "
    f"{pde_test_correlation:.8f}"
)

True PDE Solution vs. PINN
MSE(V_true, V_PINN)0.00141383
cor(V_true, V_PINN): 0.99999960


In [70]:
# Save results for 03 visualization

n_epochs = 30000

# Keep only the FIRST training run
history_first_run = {
    key: values[:n_epochs]
    for key, values in history.items()
}


# Save everything needed for Notebook 03
checkpoint_vis = {

    # Training history
    "history": history_first_run,

    # Trained model
    "model_state_dict": model.state_dict(),

    # Problem domain
    "t_min": t_min,
    "t_max": t_max,
    "S_min": S_min,
    "S_max": S_max,

    # Black-Scholes settings
    "K": K,
    "T": T,

    # True parameters used for synthetic data
    "r_true": 0.05,
    "sigma_true": 0.20,

    # Network architecture
    "hidden_width": 64,
    "hidden_layers": 4
}


torch.save(
    checkpoint_vis,
    "european_visualization_checkpoint.pt"
)

print("Visualization checkpoint saved.")

Visualization checkpoint saved.
